# PCL Binary Classification — DeBERTa-v3-base
SemEval 2022 Task 4 Subtask 1: Patronising and Condescending Language Detection

**Architecture:** DeBERTa-v3-base → Attention Pooling → Multi-sample Dropout → 2-layer Head → Focal Loss

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, classification_report

# === Config ===
CONFIG = {
    "model_name": "microsoft/deberta-v3-base",
    "max_length": 128,
    "batch_size": 16,
    "num_epochs": 5,
    "encoder_lr": 2e-5,
    "head_lr": 1e-3,
    "num_dropout_samples": 5,
    "dropout_rate": 0.1,
    "seed": 42,
}

torch.manual_seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# === Data Loading ===
TSV_PATH = "NLPLabs-2024/Dont_Patronize_Me_Trainingset/dontpatronizeme_pcl.tsv"
TRAIN_SPLIT = "dontpatronizeme/semeval-2022/practice splits/train_semeval_parids-labels.csv"
DEV_SPLIT = "dontpatronizeme/semeval-2022/practice splits/dev_semeval_parids-labels.csv"
TEST_PATH = "dontpatronizeme/semeval-2022/TEST/task4_test.tsv"

# Load main dataset
df = pd.read_csv(
    TSV_PATH, sep="\t", header=None,
    names=["par_id", "art_id", "keyword", "country_code", "text", "label"],
    quoting=3, skiprows=4,
)
df["binary_label"] = (df["label"] >= 2).astype(int)
df = df.dropna(subset=["text"])

# Apply train/dev splits
train_ids = pd.read_csv(TRAIN_SPLIT)
dev_ids = pd.read_csv(DEV_SPLIT)

train_df = df[df["par_id"].isin(train_ids["par_id"])].reset_index(drop=True)
dev_df = df[df["par_id"].isin(dev_ids["par_id"])].reset_index(drop=True)

# Load test set (no labels)
test_df = pd.read_csv(
    TEST_PATH, sep="\t", header=None,
    names=["par_id", "art_id", "keyword", "country_code", "text"],
    quoting=3,
)
test_df = test_df.dropna(subset=["text"])

n_no_pcl = (train_df["binary_label"] == 0).sum()
n_pcl = (train_df["binary_label"] == 1).sum()

print(f"Train: {len(train_df)} total | PCL: {n_pcl} | No PCL: {n_no_pcl} | Ratio: {n_no_pcl/n_pcl:.1f}:1")
print(f"Dev:   {len(dev_df)} total | PCL: {dev_df['binary_label'].sum()} | No PCL: {(dev_df['binary_label']==0).sum()}")
print(f"Test:  {len(test_df)} examples (no labels)")

In [ ]:
# === Dataset Class ===
class PCLDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }

In [ ]:
# === Model Architecture ===

class AttentionPooling(nn.Module):
    """Learnable attention-weighted pooling over token representations."""

    def __init__(self, hidden_size):
        super().__init__()
        self.attention = nn.Linear(hidden_size, 1)

    def forward(self, hidden_states, attention_mask):
        scores = self.attention(hidden_states).squeeze(-1)  # (batch, seq_len)
        scores = scores.masked_fill(attention_mask == 0, float("-inf"))
        weights = torch.softmax(scores, dim=-1)  # (batch, seq_len)
        pooled = torch.bmm(weights.unsqueeze(1), hidden_states).squeeze(1)
        return pooled, weights


class PCLClassifier(nn.Module):
    """
    DeBERTa-v3-base with:
    - Attention pooling
    - Multi-sample dropout (K=5 during training)
    - Two-layer classification head
    """

    def __init__(self, model_name, num_labels=2, dropout_rate=0.1, num_dropout_samples=5):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        self.attention_pool = AttentionPooling(hidden_size)
        self.num_dropout_samples = num_dropout_samples
        self.dropouts = nn.ModuleList([nn.Dropout(dropout_rate) for _ in range(num_dropout_samples)])

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, num_labels),
        )

        self.last_attn_weights = None

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        hidden_states = outputs.last_hidden_state

        pooled, attn_weights = self.attention_pool(hidden_states, attention_mask)
        self.last_attn_weights = attn_weights.detach()

        if self.training:
            logits = torch.stack(
                [self.classifier(dropout(pooled)) for dropout in self.dropouts],
                dim=0,
            ).mean(dim=0)
        else:
            logits = self.classifier(self.dropouts[0](pooled))

        return logits

In [ ]:
# === Focal Loss ===

class FocalLoss(nn.Module):
    """FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)"""

    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        probs = F.softmax(logits, dim=-1)
        targets_one_hot = F.one_hot(targets, num_classes=logits.size(-1)).float()
        p_t = (probs * targets_one_hot).sum(dim=-1)
        focal_weight = (1 - p_t) ** self.gamma
        ce_loss = -torch.log(p_t + 1e-8)
        loss = focal_weight * ce_loss

        if self.alpha is not None:
            alpha_t = (self.alpha.to(targets.device) * targets_one_hot).sum(dim=-1)
            loss = alpha_t * loss

        return loss.mean()

In [ ]:
# === Training & Evaluation ===

def evaluate(model, data_loader, device):
    """Evaluate model. Returns F1, predictions, and P(PCL) probabilities."""
    model.eval()
    all_preds, all_probs, all_labels = [], [], []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"]

            logits = model(input_ids, attention_mask)
            probs = torch.softmax(logits, dim=-1)[:, 1]
            preds = (probs > 0.5).long()

            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.numpy())

    f1 = f1_score(all_labels, all_preds, pos_label=1)
    return f1, np.array(all_preds), np.array(all_probs)


def train_model(model, train_loader, dev_loader, device, config, n_no_pcl, n_pcl):
    """Train with gradual unfreezing, focal loss, and per-epoch dev eval."""

    # Focal loss with inverse-frequency class weights
    total = n_no_pcl + n_pcl
    alpha = torch.tensor([n_pcl / total, n_no_pcl / total])  # higher weight for PCL
    criterion = FocalLoss(alpha=alpha, gamma=2.0)

    # Separate learning rates
    encoder_params = list(model.encoder.parameters())
    head_params = list(model.attention_pool.parameters()) + list(model.classifier.parameters())

    optimizer = AdamW([
        {"params": encoder_params, "lr": config["encoder_lr"]},
        {"params": head_params, "lr": config["head_lr"]},
    ], weight_decay=0.01)

    total_steps = len(train_loader) * config["num_epochs"]
    warmup_steps = int(total_steps * 0.1)
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    # Freeze encoder for epoch 1
    for param in model.encoder.parameters():
        param.requires_grad = False

    best_f1 = 0.0
    best_model_state = None

    for epoch in range(config["num_epochs"]):
        if epoch == 1:
            print("Unfreezing encoder...")
            for param in model.encoder.parameters():
                param.requires_grad = True

        model.train()
        total_loss = 0

        for batch_idx, batch in enumerate(train_loader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()

            if batch_idx % 50 == 0:
                print(f"  Epoch {epoch+1}, Batch {batch_idx}/{len(train_loader)}, Loss: {loss.item():.4f}")

        avg_loss = total_loss / len(train_loader)
        dev_f1, _, _ = evaluate(model, dev_loader, device)
        print(f"Epoch {epoch+1}/{config['num_epochs']} — Train Loss: {avg_loss:.4f} — Dev F1: {dev_f1:.4f}")

        if dev_f1 > best_f1:
            best_f1 = dev_f1
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            print(f"  *** New best model! F1 = {best_f1:.4f} ***")

    model.load_state_dict(best_model_state)
    return model, best_f1

In [ ]:
# === Threshold Tuning ===

def find_optimal_threshold(probs, labels, thresholds=np.arange(0.1, 0.9, 0.01)):
    """Sweep thresholds to maximise F1 on the PCL class."""
    best_f1 = 0
    best_threshold = 0.5

    for t in thresholds:
        preds = (probs >= t).astype(int)
        f1 = f1_score(labels, preds, pos_label=1)
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = t

    print(f"Optimal threshold: {best_threshold:.2f} (F1 = {best_f1:.4f})")
    print(f"Default 0.5 threshold F1: {f1_score(labels, (probs >= 0.5).astype(int), pos_label=1):.4f}")
    return best_threshold

In [ ]:
# === Run Training ===

tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])

train_dataset = PCLDataset(train_df["text"].tolist(), train_df["binary_label"].tolist(), tokenizer, CONFIG["max_length"])
dev_dataset = PCLDataset(dev_df["text"].tolist(), dev_df["binary_label"].tolist(), tokenizer, CONFIG["max_length"])

train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=CONFIG["batch_size"], shuffle=False)

model = PCLClassifier(
    model_name=CONFIG["model_name"],
    num_labels=2,
    dropout_rate=CONFIG["dropout_rate"],
    num_dropout_samples=CONFIG["num_dropout_samples"],
).to(device)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

model, best_f1 = train_model(model, train_loader, dev_loader, device, CONFIG, n_no_pcl, n_pcl)

In [ ]:
# === Threshold Tuning + Save Predictions ===

# Get dev probabilities from best model
dev_f1, dev_preds_default, dev_probs = evaluate(model, dev_loader, device)
dev_labels = dev_df["binary_label"].values

print("=== Threshold Tuning ===")
optimal_threshold = find_optimal_threshold(dev_probs, dev_labels)

# Apply optimal threshold to dev
dev_preds_tuned = (dev_probs >= optimal_threshold).astype(int)
tuned_f1 = f1_score(dev_labels, dev_preds_tuned, pos_label=1)
print(f"\nDev F1 (threshold=0.5):  {dev_f1:.4f}")
print(f"Dev F1 (threshold={optimal_threshold:.2f}): {tuned_f1:.4f}")

print("\n=== Classification Report (Dev, Tuned Threshold) ===")
print(classification_report(dev_labels, dev_preds_tuned, target_names=["No PCL", "PCL"]))

# Save dev.txt
with open("dev.txt", "w") as f:
    for pred in dev_preds_tuned:
        f.write(f"{pred}\n")
print(f"Saved dev.txt ({len(dev_preds_tuned)} predictions)")

# === Test set predictions ===
test_dataset = PCLDataset(
    test_df["text"].tolist(),
    [0] * len(test_df),  # dummy labels
    tokenizer,
    CONFIG["max_length"],
)
test_loader = DataLoader(test_dataset, batch_size=CONFIG["batch_size"], shuffle=False)

_, _, test_probs = evaluate(model, test_loader, device)
test_preds = (test_probs >= optimal_threshold).astype(int)

with open("test.txt", "w") as f:
    for pred in test_preds:
        f.write(f"{pred}\n")
print(f"Saved test.txt ({len(test_preds)} predictions)")

# Save model
torch.save(model.state_dict(), "best_model.pt")
print("Saved best_model.pt")